# DEC5 held-time epipolar evidence checks
Grain: one reference feature per anchor time and secondary train camera; errors summarized within spatial blocks, then per time/camera. These are correspondence errors in pixels, not face reconstruction metrics. No image, calibration, source time, or prediction is changed.
The original reference+secondary-xy residual model is invalid for causal interpretation (response leakage). Only the reference-only replacement is admissible. This notebook checks the retained audit and independently recomputes the fourth-time error summaries; exact standalone checks live beside `held_time_findings.json` in `audit_held_time_evidence.py`.

In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/diagnostics')
findings = json.loads((root/'temporal_source_correspondence/held_time_findings.json').read_text())
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
for path, expected in findings['retained_hashes'].items():
    assert digest(path) == expected, path
assert findings['audit_status'] == 'pass_with_explicit_supersession'
assert not findings['temporal_promotion'] and not findings['calibrated_rig_exported']
print('Verified retained hashes:', len(findings['retained_hashes']))
print('Source EXRs:', findings['source_EXRs_verified'], 'Original raw depths:', findings['raw_depth_hashes_verified'])
print('Independent sheets viewed:', findings['independent_native_sheets_inspected'])


Verified retained hashes: 166
Source EXRs: 52 Original raw depths: 62
Independent sheets viewed: 2


In [2]:
held = json.loads((root/'temporal_source_correspondence_001219/independent_holdout_unique_xy.json').read_text())
assert not set(held['fit_time_windows']) & set(held['held_time_windows'])
print('Camera | points | frozen | essential | fundamental (block median absolute pixels)')
for row in held['results']:
    records = row['held_records']
    keys = [(r['frame_id'], r['reference_index']) for r in records]
    assert len(keys) == len(set(keys))
    a = np.c_[[r['a'] for r in records], np.ones(len(records))]
    b = np.c_[[r['b'] for r in records], np.ones(len(records))]
    groups = [tuple(r['group']) for r in records]
    values = []
    for matrix, expected in [(row['frozen_matrix'], row['frozen']), *[(m['matrix'], m['held']) for m in row['models']]]:
        lines = a @ np.asarray(matrix).T
        error = abs(np.sum(b*lines, axis=1)/np.linalg.norm(lines[:, :2], axis=1))
        assert np.isfinite(error).all()
        blocks = [np.median([e for e, g in zip(error, groups) if g == block]) for block in set(groups)]
        value = float(np.median(blocks))
        assert np.isclose(value, expected['block_median_absolute_error'], atol=1e-10)
        values.append(round(value, 6))
    print(row['secondary_camera'], len(records), *values, sep=' | ')


Camera | points | frozen | essential | fundamental (block median absolute pixels)
E004_B005_1210I7 | 240 | 0.627138 | 0.409409 | 0.17418
G004_B005_1210FG | 214 | 0.405746 | 0.417887 | 0.322371


## Interpretation and limits
Free fundamental matrices transfer better than the pinned rig on these two pairs and four anchor times. Fixed-intrinsics essential fits are less consistent. That supports a separately authorized common-rig calibration experiment; it does not establish a unique calibration/timing cause, solve a joint 62-camera rig, or demonstrate a seam-free render. Repetitive fabric/hair, specular localization and limited scene geometry remain caveats. No new PSNR/SSIM/LPIPS was computed here. The every-40th-frame reconstruction/fly-through gate remains unpassed.